In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from graph_tool.all import *
import scipy as sp
import sklearn

In [394]:
gene_map = {0:"I",
            1:"I",
            2:"I",
            3:"I",
            4:"O",
            5:"O",
            6:"O",
            7:"O",
            8:"S",
            9:"S",
            10:"S",
            11:"S",
            12:"S",
            13:"S",
            14:"HK",
            15:"HK"}

In [395]:
chrom_gene_type = [[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
                   [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]]
#chrom_gene_type = np.array(chrom_gene_type)
chrom_gene_mut  = [[0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0],
                   [0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0]]
#chrom_gene_mut = np.array(chrom_gene_mut)

In [396]:
class Chromosomes():
    def __init__(self, chromosomes_gene_type, chromosomes_gene_mut, gene_map):
        self.chromosomes_gene_type = [list(chrom) for chrom in chromosomes_gene_type]
        self.chromosomes_gene_mut = [list(chrom) for chrom in chromosomes_gene_mut]
        self.N_chromosomes = len(chromosomes_gene_type)
        self.gene_map = gene_map
    
    def mutate(self, mu):
        # Random gene mutation with rate mu
        n_mut = 0
        for i in range(self.N_chromosomes):
            for j in range(len(self.chromosomes_gene_type[i])):
                if np.random.rand() < mu:
                    self.chromosomes_gene_mut[i][j] = 1
                    print("mut")
                    n_mut += 1
        return n_mut
        
    
    def crossover(self, ext_chr_gt, ext_chr_gm):
        idx_chr = np.random.randint(0, self.N_chromosomes) # Random pick the index of the chromosome to crossover
        # Add new genes to the chosen chromosome
        self.chromosomes_gene_type[idx_chr].extend(ext_chr_gt)
        self.chromosomes_gene_mut[idx_chr].extend(ext_chr_gm)

    def chromosome_inglobation(self, new_chr_type, new_chr_mut):
        # Add another chromosome
        self.chromosomes_gene_type.append(new_chr_type)
        self.chromosomes_gene_mut.append(new_chr_mut)

    def get_genes_by_type(self, gene_type):
        """
        gene_type: str
            Gene type to search for, can be one between "I", "O", "S", "HK"
        """
        #apply the gene map tp chrom_gene_type
        mapped_chromosomes = [[self.gene_map[x] for x in chr] for chr in self.chromosomes_gene_type]
        restr_gene_mut_gt = []
        restr_gene_mut_gm = []
        for i in range(len(mapped_chromosomes)):
            mc = np.array(mapped_chromosomes[i])
            restr_gene_mut_gt.append(np.array(self.chromosomes_gene_type[i])[mc==gene_type])
            restr_gene_mut_gm.append(np.array(self.chromosomes_gene_mut[i])[mc==gene_type])
        return restr_gene_mut_gt, restr_gene_mut_gm

In [538]:
class Cell():
    def __init__(self, x, y, mu0, r0, chromosomes):
        self.x = x
        self.y = y
        self.mu0 = mu0
        self.mu = mu0
        self.r0 = r0
        self.r = r0
        self.chromosomes = chromosomes
    
    def update_mu(self, dmu):
        i_genes = self.chromosomes.get_genes_by_type("I")
        unique_genes = np.unique(np.concatenate(i_genes[0]))
        act_per_chr = []
        for nc in range(len(i_genes[0])):
            act_per_chr.append(np.zeros(len(unique_genes)))
            for i in range(len(unique_genes)):
                act_per_chr[nc][i] = int(any(i_genes[1][nc][i_genes[0][nc]==unique_genes[i]])) 
        act_per_chr = np.array(act_per_chr)
        act_per_gene = np.any(act_per_chr, axis=0) # ANY to have an increase in the rate of mutation, the gene has to be mutated in at least one chromosomes
        self.mu = self.mu0 + np.sum(act_per_gene)*dmu

    def update_r(self, dr):
        # Contribution of Oncogenic genes
        o_genes = self.chromosomes.get_genes_by_type("O")
        unique_genes = np.unique(np.concatenate(o_genes[0]))
        act_per_chr = []
        for nc in range(len(o_genes[0])):
            act_per_chr.append(np.zeros(len(unique_genes)))
            for i in range(len(unique_genes)):
                act_per_chr[nc][i] = int(any(o_genes[1][nc][o_genes[0][nc]==unique_genes[i]])) 
        act_per_chr = np.array(act_per_chr)
        act_per_gene_o = np.any(act_per_chr, axis=0) # ANY to have an increase in the rate of mutation, the gene has to be mutated in at least one chromosome

        #Contribution of tumor Suppressor Genes
        s_genes = self.chromosomes.get_genes_by_type("S")
        unique_genes = np.unique(np.concatenate(s_genes[0]))
        act_per_chr = []
        for nc in range(len(s_genes[0])):
            act_per_chr.append(np.zeros(len(unique_genes)))
            for i in range(len(unique_genes)):
                act_per_chr[nc][i] = int(any(s_genes[1][nc][s_genes[0][nc]==unique_genes[i]])) 
        act_per_chr = np.array(act_per_chr)
        act_per_gene_i = np.all(act_per_chr, axis=0) # ALL the difference for the tumor suppressor is that they have to be mutated in all the chromosomes
        
        self.r = self.r0 + np.sum(act_per_gene_o)*dr+ np.sum(act_per_gene_i)*dr

    def reproduce(self):
        if np.random.rand() < self.r:
            return True
        else:
            return False

    def check_cell_death(self):
        hk_genes = self.chromosomes.get_genes_by_type("HK")
        unique_genes = np.unique(np.concatenate(hk_genes[0]))
        act_per_chr = []
        for nc in range(len(hk_genes[0])):
            act_per_chr.append(np.zeros(len(unique_genes)))
            for i in range(len(unique_genes)):
                act_per_chr[nc][i] = int(all(hk_genes[1][nc][hk_genes[0][nc]==unique_genes[i]])) # if one chromosome has more than one copy of a HK gene, it suffices that at least one is not mutated
        act_per_chr = np.array(act_per_chr)
        act_per_gene_hk = np.all(act_per_chr, axis=0) # ANY if at least one of the genes in the different chromosome works, we are ok
        if np.any(act_per_gene_hk)==True:
            return True
        else:
            return False

    def update(self, p_cross=0.001, p_ingl=0.0005, dr=0.01, dmu=0.005):
        # MODIFICATION OF DNA
        # mutation
        self.chromosomes.mutate(self.mu)
        # crossover
        #if np.random.rand()<p_cross:
        #    self.chromosomes.crossover(...)
        # chromosome inglobation
        #if np.random.rand()<p_ingl:
        #    self.chromosomes.chromosome_inglobation(...)
        # update mutation rate
        self.update_mu(dmu) 

        # REPRODUCTION
        # update reproduction rate
        self.update_r(dr)
        #reproduce
        if self.check_cell_death():
            return 0 #Death
        elif self.reproduce():
            return 1 #Reproduce
        else:
            return 2 #Nothing

ch = Chromosomes(chrom_gene_type, chrom_gene_mut, gene_map)
newc_gt = [5,1]
newc_gm = [1,0]
ch.crossover(newc_gt, newc_gm)

cell = Cell(1,1,mu0=0.01,r0=0.1,chromosomes=ch)

In [508]:
cell.chromosomes.chromosomes_gene_mut

[[0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]]

In [537]:
print(cell.update(dmu=0.001))
print(cell.mu)
print(cell.r)
cell.chromosomes.chromosomes_gene_mut

mut
mut
Death
0.013000000000000001
0.14


[[1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
 [0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0]]

In [264]:
chrom_gene_mut

[[0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]

In [ ]:
class Tissue():
    def __init__(self, L, init_cell_state):
        self.L = L
        self.cells = init_cell_state
    
    def update(self):
        actions = np.zeros([self.L, self.L])
        for i in range(self.L):
            for j in range(self.L):
                actions[i,j] = self.cells[i][j].update()

In [ ]:
L = 10
N = L*L